# XNAT Upload Tool

This notebook provides an interactive interface for uploading medical imaging data (DICOM, NIfTI, etc.) to XNAT from JupyterHub.

## Features
- **Secure Authentication** - Uses your personal XNAT Alias Token (no shared passwords!)
- **Project Selection** - Dropdown list of projects you have access to
- **Flexible Upload Hierarchy** - Upload to project, subject, session, or scan level
- **Multiple File Types** - DICOM, NIfTI, CSV, text, and any other file type
- **Batch Upload** - Upload multiple files with individual verification
- **Permission Checking** - Verifies you have access before uploading
- **DICOM Upload** - Upload to prearchive with admin review
- **Upload Verification** - Confirms each file is actually in XNAT
- **Session Reuse** - Single XNAT session prevents connection leaks

## How to Get Your XNAT Alias Token
1. Log into XNAT web interface
2. Click your username (top right) → **Profile**
3. Scroll down to **Alias Tokens** section
4. Click **Create Alias Token**
5. Copy the **Alias** and **Secret** values
6. Paste them in Step 2 below

---

## Step 1: Import Required Libraries

In [ ]:
import sys
import os
from pathlib import Path

# Add the xnat-upload-tool directory to Python path
tool_dir = Path('/workspace') / os.environ.get('JUPYTERHUB_USER', '')
if tool_dir.exists():
    sys.path.insert(0, str(tool_dir))

# Also try current directory and common locations
possible_paths = [
    Path.cwd(),
    Path('/workspace'),
    Path.home() / 'xnat-upload-tool',
    Path('/home/jovyan/xnat-upload-tool'),
]
for path in possible_paths:
    if (path / 'xnat_uploader.py').exists():
        sys.path.insert(0, str(path))
        break

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Import our XNAT uploader module
try:
    from xnat_uploader import XNATUploader, XNATUploadError
    print("XNAT Uploader module loaded successfully (v3.2 - Auto File Format Detection)")
except ImportError as e:
    print(f"Error: Could not import xnat_uploader module")
    print(f"   Make sure xnat_uploader.py is in your workspace")
    print(f"   Error details: {e}")

# Initialize the uploader
uploader = XNATUploader()
print(f"XNAT Server: {uploader.xnat_url}")

# Store projects for dropdown (global)
available_projects = []

# Pre-create project dropdown (will be populated after auth)
project_dropdown = widgets.Dropdown(
    options=[('-- Authenticate First --', '')],
    value='',
    description='* Project:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

## Step 2: Enter Your XNAT Alias Token

Enter your XNAT Alias Token credentials below. These are used to authenticate with XNAT securely.

**Your alias token is personal to you** - all uploads will be logged under your XNAT account.

In [ ]:
# Create credential input widgets
alias_widget = widgets.Text(
    value='',
    description='Alias:',
    placeholder='e.g., 2777a4f6-468c-44d7-ab07-7a8c280b342c',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

secret_widget = widgets.Password(
    value='',
    description='Secret:',
    placeholder='Your secret token (hidden)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

# Validate button
validate_btn = widgets.Button(
    description='Validate Token',
    button_style='primary',
    icon='check',
    layout=widgets.Layout(width='150px')
)

# Status output
auth_output = widgets.Output()
auth_status = widgets.HTML(value='<span style="color: orange;">Not authenticated - enter your alias token above</span>')

def validate_token(btn):
    global available_projects, project_dropdown
    with auth_output:
        clear_output()
        
        alias = alias_widget.value.strip()
        secret = secret_widget.value.strip()
        
        if not alias or not secret:
            auth_status.value = '<span style="color: red;">Please enter both Alias and Secret</span>'
            return
        
        print("Validating credentials...")
        uploader.set_credentials(alias, secret)
        result = uploader.validate_credentials()
        
        if result['valid']:
            auth_status.value = f'<span style="color: green;">Authenticated as: <strong>{result["username"]}</strong></span>'
            print(f"Success! Logged in as: {result['username']}")
            
            # Fetch accessible projects
            print("\nFetching accessible projects...")
            try:
                available_projects = uploader.get_accessible_projects()
                print(f"Found {len(available_projects)} project(s) you can access")
                
                # Update project dropdown
                project_options = [('-- Select a Project --', '')] + [
                    (f"{p['id']} - {p['name']} ({p['role'] or 'Unknown role'})", p['id'])
                    for p in available_projects
                ]
                project_dropdown.options = project_options
                project_dropdown.value = ''
                
                # Show session info
                session_info = uploader.get_session_info()
                if session_info.get('connected'):
                    print(f"\nSession established (JSESSION reused for all operations)")
                    
            except XNATUploadError as e:
                print(f"Warning: Could not fetch projects: {e}")
                print("You can still enter a project ID manually.")
        else:
            auth_status.value = f'<span style="color: red;">Authentication failed: {result["error"]}</span>'
            print(f"Failed: {result['error']}")

validate_btn.on_click(validate_token)

# Display authentication form
display(HTML("<h4>XNAT Credentials</h4>"))
display(alias_widget)
display(secret_widget)
display(widgets.HBox([validate_btn, auth_status]))
display(auth_output)

## Step 3: Upload Form

Select your target project from the dropdown, then fill in optional hierarchy details.

| Fields Provided | Upload Location |
|-----------------|----------------|
| Project only | Project → Resources |
| Project + Subject | Project → Subject → Resources |
| Project + Subject + Session | Project → Subject → Session → Resources |
| Project + Subject + Session + Scan | Project → Subject → Session → Scan → Resources |

### File Type Auto-Detection
File formats are **automatically detected** from the file extension. You can upload any mix of file types in a single batch:

| Extension | Detected Format |
|-----------|----------------|
| .nii, .nii.gz | NIFTI |
| .dcm, .dicom | DICOM (→ prearchive) |
| .csv, .tsv | CSV/TSV |
| .txt, .log | TEXT |
| .json | JSON |
| .png, .jpg, .gif | Image formats |
| .pdf | PDF |
| Other | OTHER |

In [ ]:
# Manual project ID input (fallback)
project_manual = widgets.Text(
    value='',
    description='Or enter ID:',
    placeholder='Enter project ID manually if not in dropdown',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

subject_widget = widgets.Text(
    value='',
    description='Subject ID:',
    placeholder='Optional - leave empty for project-level upload',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

session_widget = widgets.Text(
    value='',
    description='Session ID:',
    placeholder='Optional - leave empty for subject-level upload',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

scan_widget = widgets.Text(
    value='',
    description='Scan ID:',
    placeholder='Optional - leave empty for session-level upload',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

# Hierarchy indicator
hierarchy_indicator = widgets.HTML(
    value='<div style="padding: 10px; background: #e8f4f8; border-radius: 5px; margin: 10px 0;"><strong>Upload to:</strong> Select a project first</div>'
)

def get_selected_project():
    """Get the selected project ID from dropdown or manual input"""
    if project_dropdown.value:
        return project_dropdown.value
    return project_manual.value.strip()

def update_hierarchy_display(change=None):
    project = get_selected_project() or '[Project]'
    subject = subject_widget.value.strip()
    session = session_widget.value.strip()
    scan = scan_widget.value.strip()
    
    if scan and session and subject:
        path = f"<strong>{project}</strong> > <strong>{subject}</strong> > <strong>{session}</strong> > <strong>{scan}</strong> > Resources"
        color = "#e8f4e8"  # green tint
    elif session and subject:
        path = f"<strong>{project}</strong> > <strong>{subject}</strong> > <strong>{session}</strong> > Resources"
        color = "#e8f4e8"
    elif subject:
        path = f"<strong>{project}</strong> > <strong>{subject}</strong> > Resources"
        color = "#f4f4e8"  # yellow tint
    else:
        path = f"<strong>{project}</strong> > Resources"
        color = "#f4e8e8"  # red tint
    
    hierarchy_indicator.value = f'<div style="padding: 10px; background: {color}; border-radius: 5px; margin: 10px 0;"><strong>Upload to:</strong> {path}</div>'

# Observe changes to update hierarchy display
project_dropdown.observe(update_hierarchy_display, names='value')
project_manual.observe(update_hierarchy_display, names='value')
subject_widget.observe(update_hierarchy_display, names='value')
session_widget.observe(update_hierarchy_display, names='value')
scan_widget.observe(update_hierarchy_display, names='value')

# Imaging modality selector (for session creation)
modality_widget = widgets.Dropdown(
    options=[
        ('MR (Magnetic Resonance)', 'xnat:mrSessionData'),
        ('CT (Computed Tomography)', 'xnat:ctSessionData'),
        ('PET (Positron Emission Tomography)', 'xnat:petSessionData'),
        ('PET-MR (Combined PET/MR)', 'xnat:petmrSessionData'),
        ('CR (Computed Radiography)', 'xnat:crSessionData')
    ],
    value='xnat:mrSessionData',
    description='Modality:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

# Scan type label (used when creating scans)
scan_type_widget = widgets.Text(
    value='OTHER',
    description='Scan Type:',
    placeholder='e.g., T1, T2, FLAIR, DWI, OTHER',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

resource_label_widget = widgets.Text(
    value='FILES',
    description='Resource Label:',
    placeholder='e.g., NIFTI, SNAPSHOTS, FILES, DATA',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

# File selection with better description
file_path_widget = widgets.Textarea(
    value='',
    description='File Paths:',
    placeholder='Enter file paths (one per line)\n\nExamples:\n/workspace/myuser/data/scan.nii\n/workspace/myuser/results/output.csv\n/workspace/myuser/logs/processing.log',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px', height='120px')
)

# Auto-detection info
auto_detect_info = widgets.HTML(
    value='<div style="padding: 8px; background: #f0f8ff; border-radius: 5px; margin: 5px 0; font-size: 12px;"><strong>File type auto-detection:</strong> The file format (NIFTI, DICOM, CSV, etc.) is automatically detected from the file extension.</div>'
)

# Verification checkbox
verify_checkbox = widgets.Checkbox(
    value=True,
    description='Verify each upload individually (recommended)',
    style={'description_width': 'initial'}
)

# Buttons
check_permissions_btn = widgets.Button(
    description='Check Permissions',
    button_style='warning',
    icon='shield',
    layout=widgets.Layout(width='180px')
)

upload_btn = widgets.Button(
    description='Upload to XNAT',
    button_style='success',
    icon='upload',
    layout=widgets.Layout(width='180px')
)

disconnect_btn = widgets.Button(
    description='Disconnect',
    button_style='danger',
    icon='sign-out',
    layout=widgets.Layout(width='120px')
)

# Output areas
permission_output = widgets.Output()
upload_output = widgets.Output()

# Update modality visibility based on session field
def on_session_change(change):
    modality_widget.disabled = not bool(change['new'].strip())

session_widget.observe(on_session_change, names='value')

# Permission check handler
def check_permissions(btn):
    with permission_output:
        clear_output()
        
        project_id = get_selected_project()
        
        if not project_id:
            print("Please select or enter a Project ID")
            return
        
        if not uploader.authenticated_user:
            print("Please validate your alias token first (Step 2)")
            return
        
        print(f"Checking permissions for project {project_id}...")
        
        try:
            result = uploader.check_permissions(project_id)
            
            if result['has_permission']:
                print(f"Permission granted!")
                print(f"   User: {result['username']}")
                print(f"   Role: {result['role']}")
            else:
                print(f"Permission denied")
                if result.get('error'):
                    print(f"   {result['error']}")
                print(f"   Contact your XNAT administrator to request access")
        
        except XNATUploadError as e:
            print(f"Error: {e}")

check_permissions_btn.on_click(check_permissions)

# Upload handler with individual file processing and auto-detection
def upload_files(btn):
    with upload_output:
        clear_output()
        
        # Validate authentication
        if not uploader.authenticated_user:
            print("Please validate your alias token first (Step 2)")
            return
        
        # Get form values
        project_id = get_selected_project()
        subject_id = subject_widget.value.strip() or None
        session_id = session_widget.value.strip() or None
        scan_id = scan_widget.value.strip() or None
        modality = modality_widget.value
        scan_type = scan_type_widget.value.strip() or 'OTHER'
        resource_label = resource_label_widget.value.strip() or 'FILES'
        verify_upload = verify_checkbox.value
        
        if not project_id:
            print("Please select or enter a Project ID")
            return
        
        # Validate hierarchy logic
        if scan_id and not session_id:
            print("Scan ID requires Session ID")
            return
        if session_id and not subject_id:
            print("Session ID requires Subject ID")
            return
        
        # Get files to upload
        files_to_upload = []
        if file_path_widget.value.strip():
            paths = [p.strip() for p in file_path_widget.value.strip().split('\n') if p.strip()]
            for path in paths:
                p = Path(path)
                if p.exists():
                    files_to_upload.append(p)
                else:
                    print(f"Warning: File not found: {path}")
        
        if not files_to_upload:
            print("Please provide valid file paths")
            return
        
        print(f"Found {len(files_to_upload)} file(s) to upload")
        for f in files_to_upload:
            size_mb = f.stat().st_size / (1024 * 1024)
            detected_format = XNATUploader.detect_file_format(f)
            print(f"   - {f.name} ({size_mb:.2f} MB) [Format: {detected_format}]")
        
        # Check permissions
        print(f"\nChecking permissions...")
        try:
            perm_result = uploader.check_permissions(project_id)
            if not perm_result['has_permission']:
                print(f"You don't have permission to upload to project {project_id}")
                return
            print(f"Permission verified (Role: {perm_result['role']})")
        except XNATUploadError as e:
            print(f"Permission check failed: {e}")
            return
        
        # Show upload location
        hierarchy = uploader._get_hierarchy_description(project_id, subject_id, session_id, scan_id)
        print(f"\nUpload location: {hierarchy}")
        print(f"Resource label: {resource_label}")
        if scan_id:
            print(f"Scan type: {scan_type}")
        
        # Perform upload - loop through each file
        print(f"\n{'='*60}")
        print(f"Starting upload of {len(files_to_upload)} file(s)...")
        print(f"{'='*60}")
        
        success_count = 0
        fail_count = 0
        results = []
        
        for i, file_path in enumerate(files_to_upload, 1):
            # Auto-detect file format for each file
            file_format = XNATUploader.detect_file_format(file_path)
            print(f"\n[{i}/{len(files_to_upload)}] Uploading: {file_path.name} (Format: {file_format})")
            
            try:
                # Check if DICOM - goes to prearchive
                if file_format == 'DICOM' and subject_id and session_id:
                    result = uploader.upload_dicom_to_prearchive(
                        dicom_files=[file_path],
                        project_id=project_id,
                        subject_id=subject_id,
                        session_id=session_id
                    )
                    print(f"   -> Uploaded to Prearchive")
                    print(f"   -> Status: SUCCESS")
                    success_count += 1
                    results.append({'file': file_path.name, 'status': 'success', 'location': 'Prearchive', 'format': file_format})
                else:
                    # Direct upload to archive
                    result = uploader.upload_file(
                        file_path=file_path,
                        project_id=project_id,
                        subject_id=subject_id,
                        session_id=session_id,
                        scan_id=scan_id,
                        resource_label=resource_label,
                        file_format=file_format,
                        session_type=modality if session_id else None,
                        scan_type=scan_type,
                        verify_upload=verify_upload
                    )
                    
                    print(f"   -> Location: {result['location']}")
                    
                    if verify_upload:
                        if result.get('verified'):
                            print(f"   -> Verification: PASSED")
                        else:
                            print(f"   -> Verification: NOT VERIFIED")
                    
                    print(f"   -> Status: SUCCESS")
                    success_count += 1
                    results.append({
                        'file': file_path.name,
                        'status': 'success',
                        'location': result['location'],
                        'verified': result.get('verified', False),
                        'format': file_format
                    })
                    
            except XNATUploadError as e:
                print(f"   -> Status: FAILED")
                print(f"   -> Error: {e}")
                fail_count += 1
                results.append({'file': file_path.name, 'status': 'failed', 'error': str(e), 'format': file_format})
        
        # Summary
        print(f"\n{'='*60}")
        print(f"UPLOAD SUMMARY")
        print(f"{'='*60}")
        print(f"Total files: {len(files_to_upload)}")
        print(f"Successful:  {success_count}")
        print(f"Failed:      {fail_count}")
        
        if fail_count > 0:
            print(f"\nFailed files:")
            for r in results:
                if r['status'] == 'failed':
                    print(f"   - {r['file']}: {r.get('error', 'Unknown error')}")
        
        # Check if any DICOM files were uploaded
        dicom_uploaded = any(r.get('format') == 'DICOM' and r['status'] == 'success' for r in results)
        if dicom_uploaded:
            print(f"\nNext steps for DICOM files:")
            print(f"   1. Go to XNAT web interface")
            print(f"   2. Navigate to Prearchive")
            print(f"   3. Review and archive your session")

upload_btn.on_click(upload_files)

# Disconnect handler
def disconnect_session(btn):
    global project_dropdown
    with upload_output:
        clear_output()
        uploader.disconnect()
        auth_status.value = '<span style="color: orange;">Disconnected - enter your alias token to reconnect</span>'
        project_dropdown.options = [('-- Authenticate First --', '')]
        project_dropdown.value = ''
        print("Disconnected from XNAT. Session closed.")

disconnect_btn.on_click(disconnect_session)

# Display form
display(HTML("<h3>Upload Location</h3>"))
display(HTML("<p>Select a project from the dropdown, or enter a project ID manually:</p>"))
display(project_dropdown)
display(project_manual)
display(subject_widget)
display(session_widget)
display(scan_widget)
display(hierarchy_indicator)

display(HTML("<h3>File Settings</h3>"))
display(modality_widget)
display(scan_type_widget)
display(resource_label_widget)

display(HTML("<h3>Files to Upload</h3>"))
display(HTML("<p>Enter one file path per line. File format is auto-detected from extension.</p>"))
display(file_path_widget)
display(auto_detect_info)
display(verify_checkbox)

display(HTML("<h3>Actions</h3>"))
display(widgets.HBox([check_permissions_btn, upload_btn, disconnect_btn]))
display(permission_output)
display(upload_output)

---

## Troubleshooting

### Common Issues:

1. **"Authentication failed"**
   - Make sure you've created an Alias Token in XNAT
   - Go to XNAT → Profile → Alias Tokens → Create
   - Copy both the Alias and Secret values exactly
   - Tokens expire - create a new one if yours is old

2. **"Permission denied"**
   - Make sure you're a Member, Collaborator, or Owner of the project
   - Contact your XNAT administrator to grant access

3. **"File not found"**
   - Use full absolute paths (e.g., `/workspace/username/data/file.nii`)
   - Check that the file exists: run `!ls -la /path/to/file` in a cell

4. **"Upload verification failed"**
   - The file was sent but not stored correctly
   - Check XNAT web interface manually
   - May indicate a permission or storage issue

5. **"Session exists with different type"**
   - The session was already created with a different modality (e.g., MR vs CT)
   - Use a different session ID, or match the modality to the existing session

6. **"Too many open sessions" / Session leaks**
   - This version uses xnatpy which properly manages sessions
   - Sessions are reused across operations
   - Click "Disconnect" when done to explicitly close the session

### File Type Auto-Detection:

File formats are automatically detected from the extension:

| Extension | Detected Format |
|-----------|----------------|
| .nii, .nii.gz | NIFTI |
| .dcm, .dicom | DICOM (to prearchive) |
| .csv | CSV |
| .tsv | TSV |
| .txt, .log | TEXT |
| .json | JSON |
| .xml | XML |
| .png, .jpg, .jpeg, .gif, .tif, .bmp | Image formats |
| .pdf | PDF |
| .zip, .tar, .gz | Archive formats |
| .mat | MATLAB |
| .py, .r | Code files |
| Other extensions | OTHER |

### Upload Hierarchy Examples:

```
# Project-level upload (project resources)
Project: 002
→ File goes to: /data/archive/projects/002/resources/FILES/

# Subject-level upload (subject resources)  
Project: 002, Subject: SUBJ001
→ File goes to: /data/archive/projects/002/subjects/SUBJ001/resources/FILES/

# Session-level upload (session resources)
Project: 002, Subject: SUBJ001, Session: SESS001
→ File goes to: /data/archive/projects/002/subjects/SUBJ001/experiments/SESS001/resources/FILES/

# Scan-level upload (scan resources)
Project: 002, Subject: SUBJ001, Session: SESS001, Scan: 1
→ File goes to: /data/archive/projects/002/subjects/SUBJ001/experiments/SESS001/scans/1/resources/FILES/
```

### Getting Help:

- Check XNAT documentation: https://wiki.xnat.org
- xnatpy documentation: https://xnat.readthedocs.io
- Contact: Australian Imaging Service support

---

**Version:** 3.2.0  
**Last Updated:** 2026-01-23  
**Changes:** Auto File Format Detection, Scan Type Field, Removed Manual File Type Dropdown